In [ ]:
# Pixelforge project configuration
from pathlib import Path

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Open Jupyter from the Pixelforge project or notebooks directory.")

PROJECT_ROOT = find_project_root()
PROJECT_DIR = PROJECT_ROOT

IMAGE_DIR = PROJECT_ROOT / "dataset" / "images"
VIDEO_DIR = PROJECT_ROOT / "dataset" / "videos"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

print("Pixelforge root:", PROJECT_ROOT)
print("Images directory:", IMAGE_DIR)
print("Videos directory:", VIDEO_DIR)


In [ ]:
from pathlib import Path
import os
import urllib.request
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = PROJECT_ROOT
DATASET_DIR = IMAGE_DIR
OUTPUT_DIR = OUTPUT_ROOT / "unit_5_object_detection"
FRAMES_DIR = OUTPUT_DIR / "extracted_frames"
MODEL_DIR = PROJECT_DIR / "models"

for folder in [OUTPUT_DIR, FRAMES_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Python:", os.sys.version)
print("OpenCV:", cv2.__version__)
print("Dataset exists:", DATASET_DIR.exists())


In [ ]:
portrait_path = DATASET_DIR / "original_image.jpg.jpg"
desk_path = DATASET_DIR / "desk_image.jpg"

portrait_bgr = cv2.imread(str(portrait_path))
desk_bgr = cv2.imread(str(desk_path))

if portrait_bgr is None:
    raise FileNotFoundError(portrait_path)
if desk_bgr is None:
    raise FileNotFoundError(desk_path)

portrait_rgb = cv2.cvtColor(portrait_bgr, cv2.COLOR_BGR2RGB)
desk_rgb = cv2.cvtColor(desk_bgr, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(portrait_rgb); ax[0].set_title("Portrait"); ax[0].axis("off")
ax[1].imshow(desk_rgb); ax[1].set_title("Desk"); ax[1].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
import cv2
import os

cascade_path = os.path.join(
    cv2.data.haarcascades,
    "haarcascade_frontalface_default.xml"
)

print("OpenCV version:", cv2.__version__)
print("Cascade directory:", cv2.data.haarcascades)
print("File exists:", os.path.exists(cascade_path))
print("File path:", cascade_path)

In [ ]:
from pathlib import Path

DATASET_DIR = IMAGE_DIR

print("Dataset folder exists:", DATASET_DIR.exists())

if DATASET_DIR.exists():
    print("\nFiles found:")
    for file in DATASET_DIR.iterdir():
        print(repr(file.name), "|", file.suffix)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

DATASET_DIR = IMAGE_DIR

# Find images using actual filenames
portrait_path = next(
    f for f in DATASET_DIR.iterdir()
    if f.name.lower().startswith("original_image")
)

desk_path = DATASET_DIR / "desk_image.jpg"

# Load images
portrait_bgr = cv2.imread(str(portrait_path))
desk_bgr = cv2.imread(str(desk_path))

assert portrait_bgr is not None, "Portrait loading failed!"
assert desk_bgr is not None, "Desk image loading failed!"

# Convert to RGB
portrait_rgb = cv2.cvtColor(portrait_bgr, cv2.COLOR_BGR2RGB)
desk_rgb = cv2.cvtColor(desk_bgr, cv2.COLOR_BGR2RGB)

# Display
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(portrait_rgb)
axes[0].set_title("Original Portrait")
axes[0].axis("off")

axes[1].imshow(desk_rgb)
axes[1].set_title("Desk Photograph")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("Both images loaded successfully!")
print("Portrait:", portrait_path.name)
print("Desk:", desk_path.name)

In [ ]:
import torch
from ultralytics import YOLO

# Reduce CPU thread usage
torch.set_num_threads(1)

# Load pretrained YOLO11 nano model
model = YOLO("yolo11n.pt")

print("YOLO model loaded successfully!")
print("PyTorch:", torch.__version__)
print("Ultralytics:", __import__("ultralytics").__version__)

In [ ]:
def detect_objects(image, image_name):

    # Resize image to maximum 640 pixels
    h, w = image.shape[:2]
    scale = min(640 / max(h, w), 1.0)

    if scale < 1:
        image = cv2.resize(
            image,
            (int(w * scale), int(h * scale))
        )

    # Perform YOLO detection on CPU
    results = model.predict(
        source=image,
        imgsz=640,
        conf=0.25,
        device="cpu",
        verbose=False
    )

    result = results[0]

    # Generate bounding boxes
    annotated = result.plot()

    # Display detection output
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("YOLO Detection - " + image_name)
    plt.axis("off")
    plt.show()

    # Extract object details
    detections = []

    for box in result.boxes:

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        detections.append({
            "Object": model.names[class_id],
            "Confidence": round(confidence, 3)
        })

    df = pd.DataFrame(
        detections,
        columns=["Object", "Confidence"]
    )

    print("Objects detected:", len(detections))
    display(df)

    # Save annotated image and CSV
    output_path = OUTPUT_DIR / f"{image_name}_yolo.png"

    cv2.imwrite(str(output_path), annotated)

    df.to_csv(
        OUTPUT_DIR / f"{image_name}_detections.csv",
        index=False
    )

    return df, annotated

In [ ]:
portrait_df, portrait_annotated = detect_objects(
    portrait_bgr,
    "portrait"
)

In [ ]:
desk_df, desk_annotated = detect_objects(
    desk_bgr,
    "desk"
)

In [ ]:
print("PORTRAIT DETECTION")
display(portrait_df)

print("DESK IMAGE DETECTION")
display(desk_df)

In [ ]:
yolo_summary = pd.DataFrame([
    {
        "Image": "Portrait",
        "Total Objects": len(portrait_df),
        "Detected Classes": ", ".join(
            portrait_df["Object"].unique()
        ) if not portrait_df.empty else "None"
    },
    {
        "Image": "Desk",
        "Total Objects": len(desk_df),
        "Detected Classes": ", ".join(
            desk_df["Object"].unique()
        ) if not desk_df.empty else "None"
    }
])

display(yolo_summary)

yolo_summary.to_csv(
    OUTPUT_DIR / "yolo_comparison.csv",
    index=False
)

print("YOLO experiment completed and saved!")

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

VIDEO_DIR = PROJECT_ROOT / 'dataset' / 'videos'

# Find available video files
video_extensions = [".mp4", ".avi", ".mov", ".mkv"]

video_files = [
    f for f in VIDEO_DIR.iterdir()
    if f.suffix.lower() in video_extensions
]

print("Available video files:")
for i, f in enumerate(video_files):
    print(i, f.name)

if not video_files:
    raise FileNotFoundError("No video found in the dataset folder.")

# Select the first video
VIDEO_PATH = video_files[0]

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError("Could not open video.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("\nSelected video:", VIDEO_PATH.name)
print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Total frames:", total_frames)
print("Duration:", round(total_frames / fps, 2), "seconds" if fps else "unknown")

ret, frame = cap.read()

if ret:
    plt.figure(figsize=(9, 5))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title("First Video Frame")
    plt.axis("off")
    plt.show()

cap.release()

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

OUTPUT_DIR = OUTPUT_ROOT / "unit_5"
FRAME_DIR = OUTPUT_DIR / "extracted_frames"

FRAME_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_PATH))

fps = cap.get(cv2.CAP_PROP_FPS)
interval = int(fps)  # one frame per second

extracted_frames = []
frame_number = 0
saved_count = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    if frame_number % interval == 0:
        timestamp = frame_number / fps
        filename = FRAME_DIR / f"frame_{saved_count:02d}.jpg"

        cv2.imwrite(str(filename), frame)

        extracted_frames.append({
            "time": timestamp,
            "frame": frame.copy(),
            "path": filename
        })

        saved_count += 1

    frame_number += 1

cap.release()

print("Extracted frames:", saved_count)
print("Saved in:", FRAME_DIR)

In [ ]:
selected = extracted_frames[::5]

fig, axes = plt.subplots(1, len(selected), figsize=(18, 5))

for ax, item in zip(axes, selected):
    ax.imshow(cv2.cvtColor(item["frame"], cv2.COLOR_BGR2RGB))
    ax.set_title(f"{item['time']:.0f} sec")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(VIDEO_PATH))

ret, previous = cap.read()
previous_gray = cv2.cvtColor(previous, cv2.COLOR_BGR2GRAY)

# Move to approximately 10 seconds
cap.set(cv2.CAP_PROP_POS_MSEC, 10000)
ret, current = cap.read()

cap.release()

current_gray = cv2.cvtColor(current, cv2.COLOR_BGR2GRAY)

# Absolute difference
difference = cv2.absdiff(previous_gray, current_gray)

# Threshold
_, motion_mask = cv2.threshold(
    difference, 25, 255, cv2.THRESH_BINARY
)

# Reduce noise
kernel = np.ones((5, 5), np.uint8)
motion_mask = cv2.morphologyEx(
    motion_mask, cv2.MORPH_OPEN, kernel
)
motion_mask = cv2.dilate(motion_mask, kernel, iterations=2)

# Display
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(cv2.cvtColor(previous, cv2.COLOR_BGR2RGB))
axes[0].set_title("Previous Frame")

axes[1].imshow(cv2.cvtColor(current, cv2.COLOR_BGR2RGB))
axes[1].set_title("Current Frame")

axes[2].imshow(difference, cmap="gray")
axes[2].set_title("Absolute Difference")

axes[3].imshow(motion_mask, cmap="gray")
axes[3].set_title("Motion Mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "frame_differencing.png", dpi=150)
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

cap = cv2.VideoCapture(str(VIDEO_PATH))

back_sub = cv2.createBackgroundSubtractorMOG2(
    history=300,
    varThreshold=32,
    detectShadows=True
)

mog_results = []
frame_index = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    mask = back_sub.apply(frame)

    # Save representative results at 5-second intervals
    if frame_index % (int(fps) * 5) == 0:
        mog_results.append((frame_index / fps, frame.copy(), mask.copy()))

    frame_index += 1

cap.release()

print("MOG2 processed frames:", frame_index)
print("Representative outputs:", len(mog_results))

In [ ]:
fig, axes = plt.subplots(len(mog_results), 2, figsize=(12, 3 * len(mog_results)))

for i, (timestamp, frame, mask) in enumerate(mog_results):
    axes[i, 0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"Original Frame – {timestamp:.0f}s")

    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title(f"MOG2 Foreground Mask – {timestamp:.0f}s")

    axes[i, 0].axis("off")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mog2_background_subtraction.png", dpi=150)
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(VIDEO_PATH))

# Select a frame around 10 seconds
cap.set(cv2.CAP_PROP_POS_MSEC, 10000)
ret, old_frame = cap.read()

# Read the next frame
ret, new_frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("Could not read video frames.")

# Resize for faster processing
old_frame = cv2.resize(old_frame, (640, 360))
new_frame = cv2.resize(new_frame, (640, 360))

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
new_gray = cv2.cvtColor(new_frame, cv2.COLOR_BGR2GRAY)

# Detect trackable feature points
feature_params = dict(
    maxCorners=100,
    qualityLevel=0.3,
    minDistance=7,
    blockSize=7
)

old_points = cv2.goodFeaturesToTrack(
    old_gray, mask=None, **feature_params
)

if old_points is None:
    raise RuntimeError("No trackable feature points detected.")

# Lucas-Kanade optical flow
lk_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        10,
        0.03
    )
)

new_points, status, error = cv2.calcOpticalFlowPyrLK(
    old_gray,
    new_gray,
    old_points,
    None,
    **lk_params
)

# Keep successfully tracked points
good_old = old_points[status.flatten() == 1]
good_new = new_points[status.flatten() == 1]

# Draw motion vectors
flow_output = new_frame.copy()

for old, new in zip(good_old, good_new):
    x1, y1 = old.ravel().astype(int)
    x2, y2 = new.ravel().astype(int)

    cv2.arrowedLine(
        flow_output,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2,
        tipLength=0.3
    )

    cv2.circle(flow_output, (x2, y2), 3, (0, 0, 255), -1)

print("Detected feature points:", len(old_points))
print("Successfully tracked points:", len(good_new))

# Display results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(cv2.cvtColor(old_frame, cv2.COLOR_BGR2RGB))
axes[0].set_title("Previous Frame")

axes[1].imshow(cv2.cvtColor(new_frame, cv2.COLOR_BGR2RGB))
axes[1].set_title("Current Frame")

axes[2].imshow(cv2.cvtColor(flow_output, cv2.COLOR_BGR2RGB))
axes[2].set_title("Lucas-Kanade Optical Flow")

for ax in axes:
    ax.axis("off")

plt.tight_layout()

cv2.imwrite(
    str(OUTPUT_DIR / "lucas_kanade_optical_flow.png"),
    flow_output
)

plt.savefig(
    OUTPUT_DIR / "lucas_kanade_comparison.png",
    dpi=150
)

plt.show()

In [ ]:
print("Lucas-Kanade Optical Flow Results")
print("---------------------------------")
print("Input video:", VIDEO_PATH.name)
print("Frame resolution:", "640 x 360")
print("Detected points:", len(old_points))
print("Tracked points:", len(good_new))
print("Tracking success rate:",
      round(len(good_new) / len(old_points) * 100, 2), "%")

In [ ]:
### YOLO Object Detection: Observations

The pretrained YOLO11n model was applied to the portrait
and desk photographs.

The model generated bounding boxes, object class labels
and confidence scores for detected objects.

Detection performance depends on object size, visibility,
overlapping objects and the pretrained model's classes.

### Conclusion

YOLO successfully demonstrates deep-learning-based
object detection on different photographic scenes.